# Final Repeated Experiment Analysis

This is the incremental analysis notebook for the final repeated experiment protocol. The current version covers **E2-CMD-BASE** only.

E2 measures command-channel latency and transient recovery with the drone stationary and without telemetry display, video decoding, Qt rendering, or flight. Three primary runs (R01-R03) are kept separate from three packet-capture diagnostic runs (D01-PCAP-D03-PCAP).

## Analysis Rules

- The independent experimental unit is one run, not one command packet.
- Captured runs are not pooled with uncaptured primary runs.
- last_command_latency_ms is the complete client-observed battery query duration, including retries and recovery.
- Median and p95 describe routine and upper-tail behavior. Maximum latency preserves transient-recovery evidence.
- A final OK after internal timeouts is a recovered transient loss, not a final failure.
- PCAP shows packets observed at the host capture interface. It cannot alone distinguish radio interference, driver receive loss, or temporary drone silence.

In [ ]:
from pathlib import Path
import csv
import math
import re
import shutil
import statistics
import subprocess

import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (11, 5.5),
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
})

def resolve_data_dir():
    candidates = [
        Path.cwd() / "final_repeated",
        Path.cwd() / "results" / "final_repeated",
        Path.cwd().parent / "results" / "final_repeated",
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate results/final_repeated.")

DATA_DIR = resolve_data_dir()
PCAP_DIR = DATA_DIR / "pcap"
FIGURE_DIR = DATA_DIR / "analysis_images"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data directory: {DATA_DIR}")
print(f"Figures directory: {FIGURE_DIR}")

In [ ]:
def read_csv_rows(path):
    with path.open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

def number(row, key, default=math.nan):
    try:
        return float(row[key])
    except (KeyError, TypeError, ValueError):
        return default

def percentile(values, q):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return float(np.percentile(values, q)) if values.size else math.nan

def sample_sd(values):
    values = [float(value) for value in values if math.isfinite(float(value))]
    return statistics.stdev(values) if len(values) > 1 else 0.0

def save_figure(fig, filename):
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=170, bbox_inches="tight")
    print(f"Saved: {path}")
    return path

csv_paths = sorted(DATA_DIR.glob("E2-CMD-BASE-*.csv"))
primary_paths = [p for p in csv_paths if re.search(r"-R\d+\.csv$", p.name)]
diagnostic_paths = [p for p in csv_paths if "-PCAP.csv" in p.name]
runs = {path.stem: read_csv_rows(path) for path in csv_paths}

print(f"Primary CSV runs: {len(primary_paths)}")
print(f"PCAP diagnostic CSV runs: {len(diagnostic_paths)}")
for path in csv_paths:
    rows = runs[path.stem]
    duration_s = number(rows[-1], "elapsed_ms") / 1000.0
    print(f"  {path.name:<34} rows={len(rows):3d} duration={duration_s:6.2f} s")

## E2 Run Summary

Each row below is one independent run. Retries, timeouts, and recoveries are sums of per-command internal counters. A command may therefore contain timeouts and still finish successfully after recovery. Steady p95 uses only commands completed in one internal attempt; recovery events remain in the raw p95 and maximum.

In [ ]:
def summarize_run(path):
    rows = runs[path.stem]
    latency = np.array([number(r, "last_command_latency_ms") for r in rows])
    attempts = np.array([number(r, "command_attempt_count", 0) for r in rows])
    steady = latency[(attempts == 1) & np.isfinite(latency)]
    final = rows[-1]
    return {
        "run": path.stem.replace("E2-CMD-BASE-", ""),
        "mode": "PCAP" if "-PCAP" in path.stem else "No PCAP",
        "samples": len(latency),
        "successes": int(number(final, "command_successes", 0)),
        "failures": int(number(final, "command_failures", 0)),
        "median_ms": float(np.median(latency)),
        "p95_ms": percentile(latency, 95),
        "max_ms": float(np.max(latency)),
        "steady_p95_ms": percentile(steady, 95),
        "retries": int(sum(number(r, "command_retry_count", 0) for r in rows)),
        "timeouts": int(sum(number(r, "command_timeout_count", 0) for r in rows)),
        "recoveries": int(sum(number(r, "command_recovery_count", 0) for r in rows)),
    }

summaries = [summarize_run(path) for path in csv_paths]
header = (
    f"{'Run':<11} {'Mode':<8} {'N':>4} {'OK':>4} {'Fail':>5} "
    f"{'Median':>8} {'p95':>8} {'Max':>8} {'Steady p95':>11} "
    f"{'Retry':>6} {'TO':>4} {'Rec':>4}"
)
print(header)
print("-" * len(header))
for item in summaries:
    print(
        f"{item['run']:<11} {item['mode']:<8} {item['samples']:>4d} "
        f"{item['successes']:>4d} {item['failures']:>5d} "
        f"{item['median_ms']:>8.1f} {item['p95_ms']:>8.1f} "
        f"{item['max_ms']:>8.1f} {item['steady_p95_ms']:>11.1f} "
        f"{item['retries']:>6d} {item['timeouts']:>4d} "
        f"{item['recoveries']:>4d}"
    )

## Command Latency Over Time

Each point is one complete battery query. The logarithmic y-axis keeps routine values around tens of milliseconds visible while showing multi-second events. Red rings mark calls requiring more than one internal attempt. Primary and PCAP runs are separated so capture overhead is not confused with baseline behavior.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, sharey=True)
groups = [
    (primary_paths, "Primary runs without packet capture"),
    (diagnostic_paths, "Diagnostic runs with packet capture"),
]
for ax, (paths, title) in zip(axes, groups):
    for path in paths:
        rows = runs[path.stem]
        elapsed = np.array([number(r, "elapsed_ms") / 1000 for r in rows])
        latency = np.array([number(r, "last_command_latency_ms") for r in rows])
        attempts = np.array([number(r, "command_attempt_count", 0) for r in rows])
        label = path.stem.replace("E2-CMD-BASE-", "")
        ax.plot(elapsed, latency, marker=".", markersize=3, linewidth=1, label=label)
        mask = attempts > 1
        ax.scatter(
            elapsed[mask], latency[mask], s=75, facecolors="none",
            edgecolors="crimson", linewidths=1.5, zorder=5
        )
    ax.set_title(title)
    ax.set_ylabel("Complete command latency (ms)")
    ax.set_yscale("log")
    ax.set_ylim(20, 5000)
    ax.legend(ncol=3, loc="upper right")
axes[-1].set_xlabel("Experiment elapsed time (s)")
fig.suptitle("E2 command latency timeline", fontsize=14)
fig.tight_layout()
save_figure(fig, "e2_command_latency_timeline.png")
plt.show()

## Packet-Capture Overhead Check

The plot uses one median and one steady-state p95 per run, preserving the run as the independent unit. Similar values between groups suggest that tcpdump did not materially change ordinary latency. With only three runs per group, this is descriptive rather than a formal equivalence test.

In [ ]:
def group_metric(mode, key):
    return [item[key] for item in summaries if item["mode"] == mode]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, key, title in [
    (axes[0], "median_ms", "Per-run median"),
    (axes[1], "steady_p95_ms", "Per-run steady-state p95"),
]:
    for x, mode, color in [
        (0, "No PCAP", "#2878B5"),
        (1, "PCAP", "#D95F02"),
    ]:
        values = group_metric(mode, key)
        offsets = np.linspace(-0.06, 0.06, len(values))
        ax.scatter(
            np.full(len(values), x) + offsets, values,
            s=65, color=color, label=mode if ax is axes[0] else None
        )
        ax.errorbar(
            x, statistics.mean(values), yerr=sample_sd(values),
            fmt="_", markersize=24, color="black", capsize=5
        )
    ax.set_xticks([0, 1], ["No PCAP", "PCAP"])
    ax.set_ylabel("Latency (ms)")
    ax.set_title(title)
axes[0].legend()
fig.suptitle("E2 routine command latency by capture condition", fontsize=14)
fig.tight_layout()
save_figure(fig, "e2_capture_effect.png")
plt.show()

for key, label in [("median_ms", "median"), ("steady_p95_ms", "steady p95")]:
    a = group_metric("No PCAP", key)
    b = group_metric("PCAP", key)
    print(
        f"{label}: no PCAP {statistics.mean(a):.1f} +/- {sample_sd(a):.1f} ms; "
        f"PCAP {statistics.mean(b):.1f} +/- {sample_sd(b):.1f} ms "
        "(mean +/- sample SD across runs)"
    )

## Internal Timing Of Delayed Commands

For commands above 200 ms, the stacked bars separate accumulated executor receive-wait time, explicit recovery, and remaining client overhead. If receive wait dominates, the CLI delay is not caused by plotting or CSV formatting.

In [ ]:
delayed = []
for path in csv_paths:
    for row in runs[path.stem]:
        latency = number(row, "last_command_latency_ms")
        if latency > 200:
            recv_wait = number(row, "command_executor_recv_wait_total_ms", 0)
            recovery = number(row, "command_recovery_ms", 0)
            delayed.append({
                "label": (
                    f"{path.stem.replace('E2-CMD-BASE-', '')}\n"
                    f"{number(row, 'elapsed_ms') / 1000:.1f}s"
                ),
                "latency": latency,
                "recv": recv_wait,
                "recovery": recovery,
                "other": max(0, latency - recv_wait - recovery),
                "attempts": int(number(row, "command_attempt_count", 0)),
            })

x = np.arange(len(delayed))
recv = np.array([d["recv"] for d in delayed])
recovery = np.array([d["recovery"] for d in delayed])
other = np.array([d["other"] for d in delayed])
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.bar(x, recv, label="Executor receive wait", color="#2878B5")
ax.bar(x, recovery, bottom=recv, label="Recovery", color="#D95F02")
ax.bar(x, other, bottom=recv + recovery, label="Other overhead", color="#7A7A7A")
for i, item in enumerate(delayed):
    ax.text(
        i, item["latency"] + 55, f"{item['attempts']} attempt(s)",
        ha="center", fontsize=8, rotation=30
    )
ax.set_xticks(x, [d["label"] for d in delayed], rotation=35, ha="right")
ax.set_ylabel("Time (ms)")
ax.set_title("E2 internal timing for commands slower than 200 ms")
ax.legend()
fig.tight_layout()
save_figure(fig, "e2_delayed_command_timing.png")
plt.show()

## Packet-Capture Evidence

These cells invoke tcpdump to read the PCAPs. In E2, outgoing UDP payload length 8 is battery?, length 7 is command, incoming length 2 is ok, and other short incoming payloads are battery values. A request is counted as unanswered when another outgoing request appears before any incoming packet.

The PCAP clock begins at the first captured SDK packet and has a small offset from CSV elapsed time. Correlation therefore uses event order and duration rather than assuming identical zero points.

In [ ]:
PCAP_LINE = re.compile(
    r"^(\d+\.\d+)\s+\?\s+(Out|In)\s+IP .*: UDP, length (\d+)"
)

def read_pcap_packets(path):
    tcpdump = shutil.which("tcpdump")
    if not tcpdump:
        raise RuntimeError("tcpdump is required for the PCAP analysis.")
    result = subprocess.run(
        [tcpdump, "-nn", "-tt", "-r", str(path), "udp port 8889"],
        check=True, capture_output=True, text=True
    )
    packets = []
    for line in result.stdout.splitlines():
        match = PCAP_LINE.match(line)
        if match:
            packets.append({
                "timestamp": float(match.group(1)),
                "direction": match.group(2),
                "length": int(match.group(3)),
            })
    origin = packets[0]["timestamp"]
    for packet in packets:
        packet["elapsed_s"] = packet["timestamp"] - origin
    return packets

def unanswered_bursts(packets):
    bursts = []
    pending = []
    for packet in packets:
        if packet["direction"] == "Out":
            pending.append(packet)
        elif pending:
            if len(pending) > 1:
                bursts.append({
                    "start_s": pending[0]["elapsed_s"],
                    "end_s": packet["elapsed_s"],
                    "duration_s": packet["elapsed_s"] - pending[0]["elapsed_s"],
                    "unanswered": len(pending) - 1,
                })
            pending = []
    return bursts

pcap_paths = sorted(PCAP_DIR.glob("E2-CMD-BASE-*-PCAP.pcap"))
pcap_runs = {path.stem: read_pcap_packets(path) for path in pcap_paths}

header = (
    f"{'PCAP':<14} {'Packets':>8} {'Out':>6} {'In':>6} "
    f"{'Unanswered':>11} {'Start':>9} {'Resumed':>9} {'Window':>9}"
)
print(header)
print("-" * len(header))
for path in pcap_paths:
    packets = pcap_runs[path.stem]
    burst = max(unanswered_bursts(packets), key=lambda b: b["duration_s"])
    outgoing = sum(p["direction"] == "Out" for p in packets)
    incoming = sum(p["direction"] == "In" for p in packets)
    unanswered = sum(b["unanswered"] for b in unanswered_bursts(packets))
    label = path.stem.replace("E2-CMD-BASE-", "")
    print(
        f"{label:<14} {len(packets):>8d} {outgoing:>6d} {incoming:>6d} "
        f"{unanswered:>11d} {burst['start_s']:>8.3f}s "
        f"{burst['end_s']:>8.3f}s {burst['duration_s']:>8.3f}s"
    )

## UDP Events Around Each Blackout

Each subplot shows packets observed by tcpdump. The shaded interval starts with the first request in a sequence lacking an immediate response and ends when incoming traffic resumes. Repeated outgoing battery requests demonstrate that the application continued sending. The later command/ok exchange is SDK recovery.

In [ ]:
event_y = {
    "battery? sent": 3,
    "command sent": 2,
    "battery value received": 1,
    "ok received": 0,
}
event_color = {
    "battery? sent": "#2878B5",
    "command sent": "#7A3E9D",
    "battery value received": "#2E8B57",
    "ok received": "#D95F02",
}

def packet_event(packet):
    if packet["direction"] == "Out":
        return "battery? sent" if packet["length"] == 8 else "command sent"
    return "ok received" if packet["length"] == 2 else "battery value received"

fig, axes = plt.subplots(len(pcap_paths), 1, figsize=(12, 8))
if len(pcap_paths) == 1:
    axes = [axes]
for ax, path in zip(axes, pcap_paths):
    packets = pcap_runs[path.stem]
    burst = max(unanswered_bursts(packets), key=lambda b: b["duration_s"])
    start, end = max(0, burst["start_s"] - 5), burst["end_s"] + 5
    for event in event_y:
        selected = [
            p for p in packets
            if packet_event(p) == event and start <= p["elapsed_s"] <= end
        ]
        ax.scatter(
            [p["elapsed_s"] for p in selected],
            [event_y[event]] * len(selected),
            s=38, color=event_color[event], label=event, zorder=3
        )
    ax.axvspan(
        burst["start_s"], burst["end_s"],
        color="#EE5858", alpha=0.18, label="No incoming reply observed"
    )
    ax.set_yticks(list(event_y.values()), list(event_y.keys()))
    ax.set_xlim(start, end)
    ax.set_title(path.stem.replace("E2-CMD-BASE-", ""))
    ax.set_xlabel("Seconds from first captured SDK packet")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("E2 command-channel packets around each interruption", y=1.06)
fig.tight_layout()
save_figure(fig, "e2_pcap_blackout_timeline.png")
plt.show()

## Current E2 Interpretation

The next output separates direct observations from inference. Operating-system Wi-Fi logs are still needed to distinguish interface/driver events from radio or drone-side silence.

In [ ]:
all_failures = sum(item["failures"] for item in summaries)
all_recoveries = sum(item["recoveries"] for item in summaries)
recovery_times = []
for path in csv_paths:
    recovery_times.extend(
        number(row, "elapsed_ms") / 1000
        for row in runs[path.stem]
        if number(row, "command_attempt_count", 0) > 1
    )
bursts = [
    burst
    for packets in pcap_runs.values()
    for burst in unanswered_bursts(packets)
]
durations = [burst["duration_s"] for burst in bursts]

print("Evidence-supported findings")
print("---------------------------")
print(
    f"1. The six runs contain {all_failures} final failures and "
    f"{all_recoveries} completed recovery events."
)
print(
    "2. Multi-attempt commands completed at: "
    + ", ".join(f"{value:.1f} s" for value in recovery_times)
    + "."
)
print(
    f"3. The PCAPs contain {len(bursts)} request bursts with missing "
    f"immediate replies; windows range from {min(durations):.3f} "
    f"to {max(durations):.3f} s."
)
print(
    "4. Outgoing requests are present at the host interface while "
    "incoming command responses are absent until recovery."
)
print(
    "5. This rules out failure to call the UDP send path as the cause "
    "of the captured timeouts. It does not distinguish drone, radio, "
    "Wi-Fi driver, or host receive loss below the capture point."
)
print(
    "6. Delayed single-attempt responses in uncaptured runs remain "
    "valid latency observations, but CSV alone cannot localize them."
)

## Next Increment

As E3-E10 become available, each experiment should receive its own section. Comparisons should be limited to shared questions: E3 versus E4 isolates video workload, and E4 versus E5 isolates Qt workload. PCAP analysis should always remain paired with its matching CSV.